## Traffic Violations
Metadata Updated: August 9, 2018

This dataset contains traffic violation information from all electronic traffic violations issued in the County. Any information that can be used to uniquely identify the vehicle, the vehicle owner or the officer issuing the violation will not be published.

Update Frequency: Daily

The whole dataset can be downloaded from https://catalog.data.gov/dataset/traffic-violations-56dda

In [ ]:
import pandas as pd

# import  holoviews as hv
# from holoviews import opts
# hv.extension('bokeh')

import numpy as np

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
# This is needed in order to reveal the plots in the notebook
output_notebook()

from datetime import datetime as dt

# import missingno

from dotenv import load_dotenv
import os 


In [ ]:
load_dotenv("env_variables")
DATADIR = os.getenv("DATADIR")
DATADIR

In [ ]:
df = pd.read_csv(os.path.join(DATADIR, 'traffic-violations-small.csv'), parse_dates=['Date Of Stop','Time Of Stop'])  
df.head().T

### Select daily traffic violations by gender


In [ ]:
# let's create a new pandas dataframe
new_df = pd.DataFrame() 

# add the gender column
new_df['gender'] = df.Gender

# add time as it's index and rename it
new_df.index = df['Time Of Stop'].astype('datetime64[ns]')
new_df.index.rename('time', inplace=True)

In [ ]:
# to get the hourly rates/counts, we need to resample our data with an hourly interval and sum it
new_df = new_df.resample('h').sum()

# and count the "M"-s and "F"-s as strings
new_df['Male'] = new_df.gender.apply(lambda x: x.count('M'))
new_df['Female'] = new_df.gender.apply(lambda x: x.count('F'))
new_df['Total'] = new_df.gender.apply(lambda x: x.count(''))

# Reset index so the datetime index becomes a column named 'time' for Bokeh
new_df = new_df.reset_index()
# Use bokeh type for storing the dataframe
source = ColumnDataSource(new_df)

## Plot number of violations by male only

In [ ]:
# Plot it as a curve
fig = figure(
             tools='pan,wheel_zoom,box_select,reset', # add interactivity tools
             x_axis_type='datetime', # Set the x-axis to datetime type
             
             # Add the necessary informations to the plot
             title="Number of traffic violations committed by male drivers ")  # add title

# add a curve plot form the data stored now in 'source'
fig.line('time', 'Male',source=source)

# We can add info later as well
fig.xaxis.axis_label = 'Date'
fig.yaxis.axis_label = '# violations'

show(fig)


## Plot number of violations by both male/female drivers

* add legend
* add hover tool

In [ ]:
# Plot it as a curve
fig = figure(
             tools='pan,wheel_zoom,box_select,reset, hover', # add interactivity tools
             x_axis_type='datetime', # Set the x-axis to datetime type
             
             # Add the necessary informations to the plot
             title="Number of traffic violations committed by drivers (not normalized)")  # add title

# Now that we have multiple lines we will need legend as well
# add a curve plot form the data stored now in 'source'
fig.line('time', 'Male',source=source, line_width=3, legend_label="M")
# add another line
fig.line('time', 'Female',source=source, line_width=3, line_color='red', legend_label="F", )

# We can add info later as well
fig.xaxis.axis_label = 'Date'
fig.yaxis.axis_label = '# violations'


show(fig)